# Semi-Automating KB File Download

This notebook automates the file preparation and download process for KB ingestion.

## Workflow
1. Prepare the master metadata Excel file.
2. Manually bulk-download SharePoint folders to the local machine because SharePoint requires user authentication.
3. Index the downloaded local files and join their metadata to the master file list.
4. Remap document categories using `category_mapping_table.xlsx`.
5. Copy/download files into the required destination structure and export `metadata.xlsx`.

> **Manual step:** SharePoint bulk download must be completed before running the local-file indexing step.

In [ ]:
import os
import re
import shutil
import urllib.parse
from datetime import datetime

import pandas as pd
import requests


## 1. Helper functions

In [ ]:
def extract_domain(url: str) -> str:
    """Extract the network location/domain from a URL."""
    parsed_url = urllib.parse.urlparse(url)
    return parsed_url.netloc


def list_all_files(file_path: str) -> list[str]:
    """Return all files under a directory, including files in subfolders."""
    if not os.path.isdir(file_path):
        raise ValueError("Provided path is not a directory or does not exist.")

    file_list = []
    for root, _, files in os.walk(file_path):
        for file_name in files:
            file_list.append(os.path.join(root, file_name))
    return file_list


def split_filepath_into_subfolders(file_path: str) -> list[str]:
    """Split a filepath into its parent folder components."""
    subfolders = []
    while True:
        file_path, folder = os.path.split(file_path)
        if folder:
            subfolders.insert(0, folder)
        else:
            if file_path:
                subfolders.insert(0, file_path)
            break
    return subfolders


def get_metadata(source_file: str) -> dict:
    """Return metadata for a local file."""
    file_name = os.path.basename(source_file)
    modified_time = os.path.getmtime(source_file)
    file_size = os.path.getsize(source_file)
    file_extension = os.path.splitext(source_file)[1]
    subfolders = split_filepath_into_subfolders(source_file)

    return {
        "file_name": file_name,
        "modified_date": datetime.fromtimestamp(modified_time),
        "file_size": file_size,
        "file_extension": file_extension,
        "filepath": source_file,
        "filepath_N1": subfolders[-2] if len(subfolders) >= 2 else None,
        "filepath_N2": subfolders[-3] if len(subfolders) >= 3 else None,
    }


def extract_id_and_decode(url: str) -> str | None:
    """Extract and URL-decode a document identifier/filename from a URL."""
    url = url.replace("%25u2013", "-").replace("%2E", ".")
    parsed_url = urllib.parse.urlparse(url)
    query_params = urllib.parse.parse_qs(parsed_url.query)

    decoded_id = None
    if "id" in query_params:
        decoded_id = urllib.parse.unquote(query_params["id"][0])
    elif "file" in query_params:
        decoded_id = urllib.parse.unquote(query_params["file"][0])

    if decoded_id:
        return decoded_id

    decoded_url = urllib.parse.unquote(url)
    extension_pattern = r"\.(pdf|pptx|ppt|docx|doc|xlsx|xls)(?:$|[?#])"
    match = re.search(extension_pattern, decoded_url, flags=re.IGNORECASE)
    if match:
        extension = match.group(1)
        filename = decoded_url.split("/")[-1].split("?")[0].split("#")[0]
        if filename.lower().endswith(f".{extension.lower()}"):
            return filename

    return None


def download_file(url: str, save_path: str) -> bool:
    """Download a file from the DBS intranet endpoint used by the source notebook."""
    if "go.mydbs.net" not in url:
        return False

    response = requests.get(url, verify=False, timeout=60)
    if response.status_code != 200:
        return False

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "wb") as file:
        file.write(response.content)
    return True


## 2. Load the master metadata file

Expected input: `KBdocs metadata template MASTER.xlsx`, sheet `sheet1`.

In [ ]:
MASTER_FILE = "KBdocs metadata template MASTER.xlsx"
MASTER_SHEET = "sheet1"

df_info = pd.read_excel(MASTER_FILE, sheet_name=MASTER_SHEET, header=None)

# The source notebook treats the first non-empty row as the header and
# then keeps records with a non-empty Document name.
non_empty_rows = df_info.dropna(how="all")
if non_empty_rows.empty:
    raise ValueError("The master metadata file is empty.")

header_row = non_empty_rows.index[0]
df_info.columns = df_info.iloc[header_row]
df_info = df_info.iloc[header_row + 1 :].reset_index(drop=True)
df_info = df_info.loc[:, df_info.columns.notna()]

if "Document name" not in df_info.columns:
    raise KeyError("Expected column 'Document name' was not found.")

df_info = df_info[df_info["Document name"].notna()].reset_index(drop=True)
df_info.head(1)


## 3. Prepare URL and SharePoint metadata

In [ ]:
DOCUMENT_URL_COLUMN = "Hyperlink of the document (in intranet/sharepoint/mega etc)"

if DOCUMENT_URL_COLUMN not in df_info.columns:
    raise KeyError(f"Expected column '{DOCUMENT_URL_COLUMN}' was not found.")

df_info["proc_unique_id"] = df_info.apply(
    lambda row: f"{str(row['Document name']).lower().strip()}_{str(row.get('S/N', '')).lower().strip()}",
    axis=1,
)
df_info["proc_name"] = df_info["Document name"].apply(
    lambda value: str(value).lower().replace(" ", "").strip()
)
df_info["proc_url"] = df_info[DOCUMENT_URL_COLUMN]
df_info["proc_url_type"] = df_info["proc_url"].apply(lambda value: extract_domain(str(value)))
df_info["proc_url_filename"] = df_info["proc_url"].apply(
    lambda value: extract_id_and_decode(str(value))
)

# SharePoint requires interactive authentication, so the source workflow
# prepares a path for manual bulk download before local indexing.
df_info["proc_url_manualbulkdownload"] = None
sharepoint_mask = df_info["proc_url_type"].eq("dbs1bank.sharepoint.com")
df_info.loc[sharepoint_mask, "proc_url_manualbulkdownload"] = df_info.loc[
    sharepoint_mask, "proc_url"
].apply(
    lambda url: f"{str(url).split('/sites/')[-1].split('/')[0]} Documents"
    if "/sites/" in str(url)
    else None
)

df_info["proc_sharepoint_filepath"] = None


## 4. Manual SharePoint bulk download

Use the values in `proc_url_manualbulkdownload` to identify the SharePoint folders/files that need to be copied to the local machine.

After the manual download is complete, place the downloaded files under the local `data/` directory and populate `proc_sharepoint_filepath` where an explicit SharePoint path is available.

In [ ]:
df_info.loc[
    df_info["proc_url_manualbulkdownload"].notna(),
    ["proc_url_manualbulkdownload"],
].drop_duplicates()


## 5. Index locally downloaded files

In [ ]:
SOURCE_FOLDER = os.path.join(os.getcwd(), "data")

file_list = list_all_files(SOURCE_FOLDER)
file_info = []

for current_filepath in file_list:
    try:
        metadata = get_metadata(current_filepath)
        file_info.append(metadata)
    except OSError as exc:
        print(f"Skipping {current_filepath}: {exc}")

file_info = pd.DataFrame(file_info)
file_info.columns = [f"proc_sharepoint_local_{column}" for column in file_info.columns]

df_info = pd.merge(
    df_info,
    file_info,
    left_on="proc_sharepoint_file_name",
    right_on="proc_sharepoint_local_file_name",
    how="left",
)


## 6. Remap document categories

Input: `category_mapping_table.xlsx`, sheet `mapping table`.

In [ ]:
CATEGORY_MAPPING_FILE = "category_mapping_table.xlsx"
CATEGORY_MAPPING_SHEET = "mapping table"

df_map = pd.read_excel(CATEGORY_MAPPING_FILE, sheet_name=CATEGORY_MAPPING_SHEET)
df_map = df_map.rename(
    columns={
        "Category_BEFORE": "Category",
        "Category2_BEFORE": "Category2",
        "Category_AFTER": "proc_Category_mapped",
        "Category2_AFTER": "proc_Category2_mapped",
    }
)

df_info = pd.merge(
    df_info,
    df_map[["Category", "Category2", "proc_Category_mapped", "proc_Category2_mapped"]],
    on=["Category", "Category2"],
    how="left",
)


## 7. Copy/download files to the KB destination

In [ ]:
BASE_DESTINATION_FOLDER = os.path.join("output", "local")

df_info["proc_filename"] = None
df_info["proc_status"] = 0
df_info["proc_destination_filepath"] = None

for current_index in range(df_info.shape[0]):
    print(f"Processing {current_index}")
    current = df_info.loc[current_index]

    category = current.get("proc_Category_mapped")
    category2 = current.get("proc_Category2_mapped")

    if pd.isna(category):
        category = "Others"
    if pd.isna(category2):
        category2 = "Supporting Documents"

    sharepoint_filepath = current.get("proc_sharepoint_filepath")
    url_filename = current.get("proc_url_filename")

    if pd.notna(sharepoint_filepath):
        filename = os.path.basename(str(sharepoint_filepath))
    elif pd.notna(url_filename):
        filename = str(url_filename)
    else:
        # No filename can be derived from the available metadata.
        # Manual intervention is required for this record.
        continue

    destination_folder = os.path.join(
        BASE_DESTINATION_FOLDER,
        str(category),
        str(category2),
    )
    os.makedirs(destination_folder, exist_ok=True)
    destination_path = os.path.join(destination_folder, filename)

    copied = False

    local_filepath = current.get("proc_sharepoint_local_filepath")
    if pd.notna(local_filepath) and os.path.isfile(str(local_filepath)):
        shutil.copy2(str(local_filepath), destination_path)
        copied = True
    elif current.get("proc_url_type") == "go.mydbs.net":
        copied = download_file(str(current["proc_url"]), destination_path)

    if copied:
        df_info.loc[current_index, "proc_filename"] = filename
        df_info.loc[current_index, "proc_status"] = 1
        df_info.loc[current_index, "proc_destination_filepath"] = destination_path

df_info.to_excel("metadata.xlsx", index=False)


## Notes / items requiring validation

- The original pasted notebook contains several corrupted identifiers and incomplete expressions. The cleaned version normalizes obvious syntax/typing corruption.
- The exact SharePoint URL-to-folder transformation was not fully recoverable from the pasted text; validate `proc_url_manualbulkdownload` against the actual SharePoint structure before production use.
- The original workflow references `proc_sharepoint_file_name` and `proc_sharepoint_filepath`; these columns must exist or be populated before the merge/copy steps.
- `requests.get(..., verify=False)` is retained because it appears in the source workflow. In a production environment, certificate verification should be preferred where the internal certificate chain permits it.